# 🗂️ Организатор датасета для обнаружения объектов

Этот ноутбук автоматически:
- 📂 Сканирует любую структуру папок (включая вложенные)
- 🔍 Находит все .png изображения и .txt аннотации
- ✅ Проверяет соответствие изображений и аннотаций
- 🔧 Организует правильную структуру датасета
- 🏷️ Переименовывает файлы без путаницы (train_0001.png ↔ train_0001.txt)
- 📊 Проводит полную диагностику датасета
- ⚠️ Выявляет проблемы (дубликаты, ошибки формата, неравномерное распределение)

**Результат:** Готовый датасет в правильной структуре для обучения моделей!

In [ ]:
# Установка необходимых библиотек
!pip install pillow matplotlib pandas -q

print('✅ Библиотеки установлены')

In [ ]:
# Подключение Google Drive
from google.colab import drive
drive.mount('/content/drive')

print('✅ Google Drive подключен успешно!')

In [ ]:
import os
import shutil
from pathlib import Path
from collections import defaultdict, Counter
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

print('✅ Импорты загружены')

In [ ]:
# ============================================================
# КОНФИГУРАЦИЯ
# ============================================================

# ИЗМЕНИТЕ НА СВОЙ ПУТЬ К ДАТАСЕТУ
SOURCE_DATASET_PATH = '/content/drive/MyDrive/Датасет'  # Исходный датасет
OUTPUT_DATASET_NAME = 'Датасет_Organized'  # Название выходной папки

# Автоматическое определение выходного пути
drive_root = '/content/drive/MyDrive'
OUTPUT_DATASET_PATH = os.path.join(drive_root, OUTPUT_DATASET_NAME)

# Если папка существует, добавляем суффикс
counter = 1
original_output = OUTPUT_DATASET_PATH
while os.path.exists(OUTPUT_DATASET_PATH):
    OUTPUT_DATASET_PATH = f"{original_output}_{counter}"
    counter += 1

print(f'📂 Исходный датасет: {SOURCE_DATASET_PATH}')
print(f'📂 Выходной датасет: {OUTPUT_DATASET_PATH}')
print()
print('⚠️ ВАЖНО: Убедитесь что путь к исходному датасету указан правильно!')

In [ ]:
# ============================================================
# ФУНКЦИИ ДЛЯ СКАНИРОВАНИЯ ДАТАСЕТА
# ============================================================

def find_dataset_files(root_path, split_name):
    """
    Рекурсивно ищет .png и .txt файлы в любой вложенности папок
    
    Args:
        root_path: Корневая папка (например, dataset/train)
        split_name: Название выборки ('train', 'val', 'test')
    
    Returns:
        dict: {basename: {'image': path, 'label': path}}
    """
    files_dict = {}
    
    if not os.path.exists(root_path):
        print(f'⚠️ Папка {split_name} не найдена: {root_path}')
        return files_dict
    
    # Рекурсивный поиск всех .png файлов
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.endswith('.png'):
                full_path = os.path.join(root, file)
                basename = os.path.splitext(file)[0]
                
                if basename not in files_dict:
                    files_dict[basename] = {'image': None, 'label': None}
                files_dict[basename]['image'] = full_path
    
    # Рекурсивный поиск всех .txt файлов
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.endswith('.txt'):
                full_path = os.path.join(root, file)
                basename = os.path.splitext(file)[0]
                
                if basename not in files_dict:
                    files_dict[basename] = {'image': None, 'label': None}
                files_dict[basename]['label'] = full_path
    
    return files_dict

def validate_yolo_annotation(label_path):
    """
    Проверяет формат YOLO аннотации
    
    Returns:
        tuple: (is_valid, errors, classes)
    """
    errors = []
    classes = []
    
    try:
        with open(label_path, 'r') as f:
            lines = f.readlines()
            
        if len(lines) == 0:
            errors.append('Пустой файл')
            return False, errors, classes
        
        for i, line in enumerate(lines, 1):
            line = line.strip()
            if not line:
                continue
            
            parts = line.split()
            if len(parts) != 5:
                errors.append(f'Строка {i}: ожидается 5 значений, получено {len(parts)}')
                continue
            
            try:
                class_id, x_center, y_center, width, height = map(float, parts)
                
                # Проверка диапазонов
                if not (0 <= x_center <= 1):
                    errors.append(f'Строка {i}: x_center вне диапазона [0, 1]')
                if not (0 <= y_center <= 1):
                    errors.append(f'Строка {i}: y_center вне диапазона [0, 1]')
                if not (0 < width <= 1):
                    errors.append(f'Строка {i}: width вне диапазона (0, 1]')
                if not (0 < height <= 1):
                    errors.append(f'Строка {i}: height вне диапазона (0, 1]')
                
                classes.append(int(class_id))
            except ValueError:
                errors.append(f'Строка {i}: невозможно преобразовать в числа')
    
    except Exception as e:
        errors.append(f'Ошибка чтения файла: {str(e)}')
        return False, errors, classes
    
    is_valid = len(errors) == 0
    return is_valid, errors, classes

print('✅ Функции сканирования загружены')

In [ ]:
# ============================================================
# СКАНИРОВАНИЕ ИСХОДНОГО ДАТАСЕТА
# ============================================================

print('🔍 Начинаю сканирование датасета...\n')

# Поиск файлов в каждой выборке
splits = {}
for split_name in ['train', 'val', 'test']:
    split_path = os.path.join(SOURCE_DATASET_PATH, split_name)
    print(f'📂 Сканирую {split_name}...')
    files_dict = find_dataset_files(split_path, split_name)
    splits[split_name] = files_dict
    print(f'   Найдено уникальных имен файлов: {len(files_dict)}')

print('\n' + '='*60)
print('📊 СТАТИСТИКА СКАНИРОВАНИЯ')
print('='*60)

total_images = 0
total_labels = 0
total_matched = 0
total_orphan_images = 0
total_orphan_labels = 0

for split_name, files_dict in splits.items():
    images = sum(1 for f in files_dict.values() if f['image'] is not None)
    labels = sum(1 for f in files_dict.values() if f['label'] is not None)
    matched = sum(1 for f in files_dict.values() if f['image'] is not None and f['label'] is not None)
    orphan_images = sum(1 for f in files_dict.values() if f['image'] is not None and f['label'] is None)
    orphan_labels = sum(1 for f in files_dict.values() if f['image'] is None and f['label'] is not None)
    
    print(f'\n{split_name.upper()}:')
    print(f'  📷 Изображений (.png): {images}')
    print(f'  📄 Аннотаций (.txt): {labels}')
    print(f'  ✅ Пар (png+txt): {matched}')
    if orphan_images > 0:
        print(f'  ⚠️ Изображений без аннотаций: {orphan_images}')
    if orphan_labels > 0:
        print(f'  ⚠️ Аннотаций без изображений: {orphan_labels}')
    
    total_images += images
    total_labels += labels
    total_matched += matched
    total_orphan_images += orphan_images
    total_orphan_labels += orphan_labels

print('\n' + '='*60)
print(f'📊 ИТОГО:')
print(f'  Всего изображений: {total_images}')
print(f'  Всего аннотаций: {total_labels}')
print(f'  Валидных пар: {total_matched}')
if total_orphan_images > 0:
    print(f'  ⚠️ Изображений без аннотаций: {total_orphan_images}')
if total_orphan_labels > 0:
    print(f'  ⚠️ Аннотаций без изображений: {total_orphan_labels}')
print('='*60)

In [ ]:
# ============================================================
# ВАЛИДАЦИЯ ФОРМАТА YOLO
# ============================================================

print('\n🔍 Проверка формата YOLO аннотаций...\n')

validation_stats = {
    'valid': 0,
    'invalid': 0,
    'errors': []
}

all_classes = []

for split_name, files_dict in splits.items():
    print(f'📂 Проверяю {split_name}...')
    
    for basename, paths in files_dict.items():
        if paths['label'] is None:
            continue
        
        is_valid, errors, classes = validate_yolo_annotation(paths['label'])
        
        if is_valid:
            validation_stats['valid'] += 1
            all_classes.extend(classes)
        else:
            validation_stats['invalid'] += 1
            validation_stats['errors'].append({
                'file': paths['label'],
                'errors': errors
            })

print('\n' + '='*60)
print('📊 РЕЗУЛЬТАТЫ ВАЛИДАЦИИ')
print('='*60)
print(f'✅ Валидных файлов: {validation_stats["valid"]}')
print(f'❌ Невалидных файлов: {validation_stats["invalid"]}')

if validation_stats['invalid'] > 0:
    print(f'\n⚠️ НАЙДЕНЫ ОШИБКИ В {validation_stats["invalid"]} ФАЙЛАХ:')
    for i, error_info in enumerate(validation_stats['errors'][:5], 1):  # Показываем первые 5
        print(f'\n{i}. Файл: {error_info["file"]}')
        for error in error_info['errors'][:3]:  # Первые 3 ошибки
            print(f'   - {error}')
    
    if len(validation_stats['errors']) > 5:
        print(f'\n   ... и еще {len(validation_stats["errors"]) - 5} файлов с ошибками')

print('='*60)

In [ ]:
# ============================================================
# ОРГАНИЗАЦИЯ ДАТАСЕТА
# ============================================================

print(f'\n🔧 Создаю организованный датасет...\n')
print(f'📂 Выходная папка: {OUTPUT_DATASET_PATH}\n')

# Создание структуры папок
for split_name in ['train', 'val', 'test']:
    os.makedirs(os.path.join(OUTPUT_DATASET_PATH, split_name, 'images'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DATASET_PATH, split_name, 'labels'), exist_ok=True)

# Копирование и переименование файлов
copy_stats = {'train': 0, 'val': 0, 'test': 0}
skipped_stats = {'train': 0, 'val': 0, 'test': 0}

for split_name, files_dict in splits.items():
    print(f'📂 Обрабатываю {split_name}...')
    
    counter = 1
    
    for basename, paths in sorted(files_dict.items()):
        # Пропускаем если нет обоих файлов
        if paths['image'] is None or paths['label'] is None:
            skipped_stats[split_name] += 1
            continue
        
        # Новое имя файла: train_0001, val_0042, test_0123
        new_basename = f"{split_name}_{counter:04d}"
        
        # Пути назначения
        dest_image = os.path.join(OUTPUT_DATASET_PATH, split_name, 'images', f'{new_basename}.png')
        dest_label = os.path.join(OUTPUT_DATASET_PATH, split_name, 'labels', f'{new_basename}.txt')
        
        # Копирование файлов
        try:
            shutil.copy2(paths['image'], dest_image)
            shutil.copy2(paths['label'], dest_label)
            copy_stats[split_name] += 1
            counter += 1
        except Exception as e:
            print(f'   ⚠️ Ошибка копирования {basename}: {e}')
            skipped_stats[split_name] += 1

print('\n' + '='*60)
print('📊 РЕЗУЛЬТАТЫ ОРГАНИЗАЦИИ')
print('='*60)

for split_name in ['train', 'val', 'test']:
    print(f'{split_name.upper()}:')
    print(f'  ✅ Скопировано пар: {copy_stats[split_name]}')
    if skipped_stats[split_name] > 0:
        print(f'  ⚠️ Пропущено: {skipped_stats[split_name]}')

total_copied = sum(copy_stats.values())
total_skipped = sum(skipped_stats.values())

print(f'\n📊 ИТОГО:')
print(f'  ✅ Всего скопировано пар: {total_copied}')
if total_skipped > 0:
    print(f'  ⚠️ Всего пропущено: {total_skipped}')
print('='*60)

print(f'\n✅ Датасет успешно организован!')
print(f'📂 Новый датасет находится в: {OUTPUT_DATASET_PATH}')

In [ ]:
# ============================================================
# ДИАГНОСТИКА ДАТАСЕТА (как в R-CNN модели)
# ============================================================

print('\n' + '='*60)
print('🔍 ПОЛНАЯ ДИАГНОСТИКА ДАТАСЕТА')
print('='*60)

# Анализ классов в каждой выборке
split_classes = {'train': [], 'val': [], 'test': []}

for split_name in ['train', 'val', 'test']:
    label_dir = os.path.join(OUTPUT_DATASET_PATH, split_name, 'labels')
    
    if os.path.exists(label_dir):
        for label_file in os.listdir(label_dir):
            if label_file.endswith('.txt'):
                label_path = os.path.join(label_dir, label_file)
                with open(label_path, 'r') as f:
                    for line in f.readlines():
                        if line.strip():
                            class_id = int(float(line.strip().split()[0]))
                            split_classes[split_name].append(class_id)

# Уникальные классы в каждой выборке
train_classes = sorted(set(split_classes['train']))
val_classes = sorted(set(split_classes['val']))
test_classes = sorted(set(split_classes['test']))
all_unique_classes = sorted(set(train_classes + val_classes + test_classes))

print(f'\n📊 КЛАССЫ В ДАТАСЕТЕ:')
print(f'  Train классы: {train_classes}')
print(f'  Val классы: {val_classes}')
print(f'  Test классы: {test_classes}')
print(f'  Всего уникальных классов: {len(all_unique_classes)}')

# Проверка проблем с классами
print(f'\n⚠️ ПРОВЕРКА ПРОБЛЕМ:')

has_problems = False

# Классы только в val, но не в train
only_in_val = set(val_classes) - set(train_classes)
if only_in_val:
    has_problems = True
    print(f'  🚨 КРИТИЧНО: Классы {sorted(only_in_val)} есть в val, но НЕТ в train!')
    print(f'     Модель не сможет обучиться на этих классах!')

# Классы только в test, но не в train
only_in_test = set(test_classes) - set(train_classes)
if only_in_test:
    has_problems = True
    print(f'  🚨 КРИТИЧНО: Классы {sorted(only_in_test)} есть в test, но НЕТ в train!')
    print(f'     Модель не сможет обучиться на этих классах!')

# Несбалансированность классов
train_counter = Counter(split_classes['train'])
if len(train_counter) > 1:
    min_count = min(train_counter.values())
    max_count = max(train_counter.values())
    imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
    
    if imbalance_ratio > 10:
        has_problems = True
        print(f'  ⚠️ ДИСБАЛАНС: Разница между классами в train более чем в {imbalance_ratio:.1f}x')
        print(f'     Минимум: {min_count}, Максимум: {max_count}')
        print(f'     Рекомендуется балансировка классов!')

# Мало данных
if len(split_classes['train']) < 100:
    has_problems = True
    print(f'  ⚠️ МАЛО ДАННЫХ: Всего {len(split_classes["train"])} объектов в train')
    print(f'     Рекомендуется минимум 100-200 объектов на класс!')

if not has_problems:
    print(f'  ✅ Проблем не обнаружено!')

# Распределение объектов по классам
print(f'\n📊 РАСПРЕДЕЛЕНИЕ ОБЪЕКТОВ ПО КЛАССАМ:')
print(f'\n  TRAIN:')
for class_id in sorted(train_counter.keys()):
    count = train_counter[class_id]
    percent = (count / len(split_classes['train'])) * 100
    print(f'    Класс {class_id}: {count:4d} объектов ({percent:5.1f}%)')

if len(val_classes) > 0:
    val_counter = Counter(split_classes['val'])
    print(f'\n  VAL:')
    for class_id in sorted(val_counter.keys()):
        count = val_counter[class_id]
        percent = (count / len(split_classes['val'])) * 100
        print(f'    Класс {class_id}: {count:4d} объектов ({percent:5.1f}%)')

if len(test_classes) > 0:
    test_counter = Counter(split_classes['test'])
    print(f'\n  TEST:')
    for class_id in sorted(test_counter.keys()):
        count = test_counter[class_id]
        percent = (count / len(split_classes['test'])) * 100
        print(f'    Класс {class_id}: {count:4d} объектов ({percent:5.1f}%)')

print('='*60)

In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ РАСПРЕДЕЛЕНИЯ
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Количество файлов в каждой выборке
axes[0, 0].bar(['Train', 'Val', 'Test'], 
               [copy_stats['train'], copy_stats['val'], copy_stats['test']],
               color=['#3498db', '#2ecc71', '#e74c3c'], alpha=0.7, edgecolor='black')
axes[0, 0].set_ylabel('Количество пар (png+txt)', fontsize=11)
axes[0, 0].set_title('Распределение файлов по выборкам', fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')

for i, (split, count) in enumerate([('train', copy_stats['train']), 
                                      ('val', copy_stats['val']), 
                                      ('test', copy_stats['test'])]):
    axes[0, 0].text(i, count, str(count), ha='center', va='bottom', 
                    fontsize=11, fontweight='bold')

# 2. Распределение объектов по классам в Train
if len(train_counter) > 0:
    classes_list = sorted(train_counter.keys())
    counts_list = [train_counter[c] for c in classes_list]
    
    axes[0, 1].bar([f'Класс {c}' for c in classes_list], counts_list,
                   color='#3498db', alpha=0.7, edgecolor='black')
    axes[0, 1].set_ylabel('Количество объектов', fontsize=11)
    axes[0, 1].set_title('Распределение классов в Train', fontsize=13, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    for i, count in enumerate(counts_list):
        axes[0, 1].text(i, count, str(count), ha='center', va='bottom',
                       fontsize=10, fontweight='bold')

# 3. Распределение объектов по классам в Val
if len(val_classes) > 0:
    val_counter = Counter(split_classes['val'])
    classes_list = sorted(val_counter.keys())
    counts_list = [val_counter[c] for c in classes_list]
    
    axes[1, 0].bar([f'Класс {c}' for c in classes_list], counts_list,
                   color='#2ecc71', alpha=0.7, edgecolor='black')
    axes[1, 0].set_ylabel('Количество объектов', fontsize=11)
    axes[1, 0].set_title('Распределение классов в Val', fontsize=13, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    for i, count in enumerate(counts_list):
        axes[1, 0].text(i, count, str(count), ha='center', va='bottom',
                       fontsize=10, fontweight='bold')

# 4. Распределение объектов по классам в Test
if len(test_classes) > 0:
    test_counter = Counter(split_classes['test'])
    classes_list = sorted(test_counter.keys())
    counts_list = [test_counter[c] for c in classes_list]
    
    axes[1, 1].bar([f'Класс {c}' for c in classes_list], counts_list,
                   color='#e74c3c', alpha=0.7, edgecolor='black')
    axes[1, 1].set_ylabel('Количество объектов', fontsize=11)
    axes[1, 1].set_title('Распределение классов в Test', fontsize=13, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    for i, count in enumerate(counts_list):
        axes[1, 1].text(i, count, str(count), ha='center', va='bottom',
                       fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print('✅ Визуализация завершена!')

In [ ]:
# ============================================================
# ФИНАЛЬНЫЙ ОТЧЕТ
# ============================================================

print('\n' + '='*60)
print('✅ ОРГАНИЗАЦИЯ ДАТАСЕТА ЗАВЕРШЕНА')
print('='*60)

print(f'\n📂 Новый датасет находится в:')
print(f'   {OUTPUT_DATASET_PATH}')

print(f'\n📊 Структура:')
print(f'   {OUTPUT_DATASET_NAME}/')
print(f'   ├── train/')
print(f'   │   ├── images/ ({copy_stats["train"]} файлов)')
print(f'   │   └── labels/ ({copy_stats["train"]} файлов)')
print(f'   ├── val/')
print(f'   │   ├── images/ ({copy_stats["val"]} файлов)')
print(f'   │   └── labels/ ({copy_stats["val"]} файлов)')
print(f'   └── test/')
print(f'       ├── images/ ({copy_stats["test"]} файлов)')
print(f'       └── labels/ ({copy_stats["test"]} файлов)')

print(f'\n📝 Схема именования файлов:')
print(f'   train_0001.png ↔ train_0001.txt')
print(f'   val_0001.png ↔ val_0001.txt')
print(f'   test_0001.png ↔ test_0001.txt')

print(f'\n🎯 Следующие шаги:')
print(f'   1. Измените путь DATA_ROOT в comparison ноутбуках на:')
print(f'      {OUTPUT_DATASET_PATH}')
print(f'   2. Запустите обучение модели!')

if has_problems:
    print(f'\n⚠️ ВНИМАНИЕ: Обнаружены проблемы в датасете!')
    print(f'   Рекомендуется исправить их перед обучением.')
    print(f'   Смотрите раздел "ПРОВЕРКА ПРОБЛЕМ" выше.')

print('\n' + '='*60)
print('🎉 Датасет готов к использованию!')
print('='*60)

## 📝 Примечания

**Что делает этот ноутбук:**
1. ✅ Рекурсивно сканирует любую структуру папок
2. ✅ Находит все .png и .txt файлы
3. ✅ Проверяет соответствие изображений и аннотаций
4. ✅ Валидирует формат YOLO
5. ✅ Создает правильную структуру dataset/train|val|test/images|labels/
6. ✅ Переименовывает файлы безопасно (train_0001.png ↔ train_0001.txt)
7. ✅ Проводит полную диагностику датасета
8. ✅ Выявляет проблемы (дисбаланс классов, ошибки формата)
9. ✅ Создает визуализации распределения

**Важно:**
- Оригинальный датасет НЕ изменяется
- Создается новая папка с организованными данными
- Соответствие изображения и аннотации гарантировано
- Совместимо со всеми comparison ноутбуками